# INTHON 2026 · Google Colab 스타터킷

Google Drive의 `starterkit`으로 **환경 설정 → 데이터 준비 → 학습 → 검증 → 제출 ZIP 생성**을 진행합니다.
기본 모델은 DINOv2와 MPNet 인코더를 동결하고 두 특징을 결합한 MLP만 학습합니다.

**시작하기**
1. `starterkit` 폴더를 Google Drive의 **내 드라이브**에 업로드합니다.
   `.venv`, `__pycache__`, `.pytest_cache`는 업로드할 필요가 없습니다.
2. 운영진에게 받은 **Train 데이터도 별도로** 업로드합니다. 이 스타터킷에는 원본 Train이 포함되어 있지 않습니다.
3. 이 노트북을 Colab으로 열고 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다.
4. 아래 경로 설정을 확인하고 셀을 위에서부터 실행합니다.

기본 폴더 배치:
```text
내 드라이브/
├── starterkit/
│   ├── starterkit_colab.ipynb
│   ├── train.py, config.json, ...
│   └── models/                    # 있으면 복사, 없으면 허용 모델 다운로드
└── data/
    └── Train/
        ├── episode_1/description.txt
        ├── episode_1/images/1.jpg, 2.jpg, ...
        └── episode_2/...
```
`Train/` 대신 `Train.zip`을 사용할 수도 있습니다. ZIP 내부가
`episode_*/...`, `Train/episode_*/...` 또는 `data/Train/episode_*/...`인 형태를 지원합니다.

학습 중 체크포인트와 기록은 `starterkit/colab_runs/<실험 이름>/`에 저장됩니다.
동봉된 `outputs/smoke`와 `sample_submission.zip`은 기능 확인용 예시입니다.
이 노트북은 새 실험을 만들어 직접 학습한 결과를 제출물로 만듭니다.

## 1. Google Drive 연결
Google 계정 연결 창에서 접근을 허용합니다. 공유 폴더를 사용할 때는
**내 드라이브에 바로가기 추가** 후 해당 경로를 아래에 입력하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. 경로와 학습 설정
`DRIVE_STARTERKIT`과 `TRAIN_SOURCE`를 자신의 Drive 경로에 맞추세요.
`TRAIN_SOURCE`는 **Train 폴더 또는 ZIP 파일**입니다.

기본값은 전체 Train, 최대 20 epoch입니다. 먼저 동작만 확인하려면
`SMOKE_TEST = True`로 바꾸면 12개 이하의 에피소드로 3 epoch만 실행합니다.
실험 이름은 자동 생성되며, 이 셀을 다시 실행하면 새 실험이 시작됩니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid

DRIVE_STARTERKIT = Path("/content/drive/MyDrive/starterkit")
TRAIN_SOURCE = Path("/content/drive/MyDrive/data/Train")  # 또는 data/Train.zip
SMOKE_TEST = False
EPOCHS = 20
IMAGE_BATCH_SIZE = 8  # T4 기본값. GPU 메모리가 부족하면 4 또는 2로 줄이세요.
TEXT_BATCH_SIZE = 16
DOWNLOAD_ZIP_TO_COMPUTER = False  # 마지막 셀에서 브라우저 다운로드도 실행

RUN_NAME = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
if SMOKE_TEST:
    RUN_NAME = "smoke_" + RUN_NAME
DRIVE_RUN = DRIVE_STARTERKIT / "colab_runs" / RUN_NAME
LOCAL_ROOT = Path("/content/inthon") / RUN_NAME
LOCAL_KIT = LOCAL_ROOT / "starterkit"
CHECKPOINT_DIR = DRIVE_RUN / "training"
EXPORT_DIR = LOCAL_ROOT / "submission"
LOCAL_ZIP = LOCAL_ROOT / f"{RUN_NAME}_submission.zip"

required = ["config.json", "train.py", "evaluate.py", "export.py", "validate.py",
            "package.py", "submission.py", "baseline_model.py", "common.py", "dataset.py",
            "download_models.py"]
missing = [name for name in required if not (DRIVE_STARTERKIT / name).is_file()]
if missing:
    raise FileNotFoundError(f"starterkit 경로 또는 파일을 확인하세요: {DRIVE_STARTERKIT} / {missing}")
if not TRAIN_SOURCE.exists():
    raise FileNotFoundError(f"Train 데이터가 없습니다. TRAIN_SOURCE를 수정하세요: {TRAIN_SOURCE}")
if not isinstance(EPOCHS, int) or EPOCHS < 1 or min(IMAGE_BATCH_SIZE, TEXT_BATCH_SIZE) < 1:
    raise ValueError("epoch와 배치 크기는 양의 정수여야 합니다.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Colab 런타임 유형을 T4 GPU로 변경한 뒤 처음부터 실행하세요.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader"], check=True)
DRIVE_RUN.mkdir(parents=True, exist_ok=False)
LOCAL_KIT.mkdir(parents=True, exist_ok=False)
print("실험:", RUN_NAME)
print("Drive 저장 위치:", DRIVE_RUN)


## 3. 대회 버전의 실행 환경 설치
채점 환경 기준일: **2026-10-31**. Python **3.12.3**, PyTorch **2.8.0 / CUDA 12.8**,
Transformers **4.57.6**와 대회에 제공되는 보조 패키지를 설치합니다.

Colab 기본 패키지와의 충돌을 줄이기 위해 `/content`에 별도 Python 환경을 만들고,
이후 학습 명령은 모두 그 환경의 `PYTHON`으로 실행합니다. **런타임 재시작은 필요하지 않습니다.**
설치에는 수 GB의 다운로드와 몇 분이 필요합니다.

이 노트북에서는 아래 버전 목록을 사용합니다. 스타터킷의 기존 `requirements.txt`를
추가 설치하지 마세요. 별도의 코드를 실행할 때도 `run_python(...)` 또는 `run_script(...)`를
사용해야 같은 환경이 적용됩니다. 일반 셀의 `import torch`는 Colab 기본 환경을 사용합니다.

In [ ]:
PYTHON_VERSION = "3.12.3"
ENV_DIR = Path("/content/inthon-python312-cu128")
TOOL_DIR = Path("/content/inthon-colab-tools")
UV = TOOL_DIR / "bin/uv"
if not UV.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
                    "--target", str(TOOL_DIR), "uv==0.13.4"], check=True)
if not (ENV_DIR / "bin/python").is_file():
    subprocess.run([str(UV), "python", "install", PYTHON_VERSION], check=True)
    subprocess.run([str(UV), "venv", "--python", PYTHON_VERSION,
                    "--seed", str(ENV_DIR)], check=True)
PYTHON = ENV_DIR / "bin/python"
subprocess.run([str(PYTHON), "-c",
                "import sys; assert sys.version_info[:3] == (3, 12, 3), sys.version"], check=True)

TORCH_PINS = ["torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128"]
PACKAGE_PINS = [
    "transformers==4.57.6", "numpy==2.5.4", "Pillow==12.3.0", "safetensors==0.8.0",
    "huggingface-hub==0.36.2", "tokenizers==0.22.2",
    "sentence-transformers==5.4.0", "einops==0.8.2", "timm==1.0.30",
    "scipy==1.18.1", "scikit-learn==1.9.1", "opencv-python-headless==4.14.0.94",
    "peft==0.18.1", "accelerate==1.15.0", "bitsandbytes==0.50.2",
    "cloudpickle==3.1.2", "joblib==1.6.0", "narwhals==2.28.0", "threadpoolctl==3.7.0",
]
PINS_PATH = DRIVE_RUN / "requirements-colab.txt"
PINS_PATH.write_text("\n".join(TORCH_PINS + PACKAGE_PINS) + "\n", encoding="utf-8")
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
                "--index-url", "https://download.pytorch.org/whl/cu128", *TORCH_PINS], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
                "-r", str(PINS_PATH)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)

ENV = os.environ.copy()
ENV.pop("PYTHONPATH", None)
ENV.pop("PYTHONHOME", None)
ENV.update({"HF_HOME": "/content/inthon-hf-cache", "HF_HUB_ENABLE_HF_TRANSFER": "0",
            "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
            "PYTHONNOUSERSITE": "1", "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"})
for name in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    ENV.pop(name, None)

def run_command(args, *, log_name=None, extra_env=None, cwd=None):
    command = [str(arg) for arg in args]
    log = (DRIVE_RUN / log_name).open("a", encoding="utf-8") if log_name else None
    try:
        with subprocess.Popen(command, cwd=cwd or LOCAL_KIT,
                              env={**ENV, **(extra_env or {})}, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    if log:
                        log.write(line)
                        log.flush()
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    finally:
        if log:
            log.close()

def run_python(source, *args, **kwargs):
    run_command([PYTHON, "-u", "-c", source, *args], **kwargs)

def run_script(script, *args, **kwargs):
    run_command([PYTHON, "-u", LOCAL_KIT / script, *args], **kwargs)

freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze", "--all"], text=True, env=ENV)
(DRIVE_RUN / "requirements-installed.txt").write_text(freeze, encoding="utf-8")
print("실행 환경 준비 완료:", PYTHON)


## 4. GPU와 주요 패키지 확인
실제 CUDA 연산, 패키지 import, bitsandbytes 4비트 연산을 검사합니다.
결과는 Drive의 `environment.json`에 남습니다. Colab의 GPU·드라이버와 채점 서버의
GPU·드라이버는 다르므로 실행 시간이나 수치가 완전히 같다는 의미는 아닙니다.

In [ ]:
run_python(r"""
import importlib, importlib.metadata as metadata, json, platform, sys
from pathlib import Path
import torch
for module in ("torchvision", "torchaudio", "transformers", "numpy", "PIL", "safetensors",
               "sentence_transformers", "einops", "timm", "scipy", "sklearn", "cv2",
               "peft", "accelerate", "bitsandbytes"):
    importlib.import_module(module)
pins = Path(sys.argv[1]).read_text().splitlines()
versions = {}
for pin in pins:
    name, expected = pin.split("==")
    versions[name] = metadata.version(name)
    assert versions[name] == expected, (name, versions[name], expected)
assert torch.cuda.is_available(), "CUDA를 사용할 수 없습니다. GPU 런타임을 확인하세요."
device = torch.device("cuda:0")
x = torch.randn(32, 64, device=device, dtype=torch.float16)
assert torch.isfinite(x @ x.T).all()
import bitsandbytes as bnb
layer = bnb.nn.Linear4bit(64, 16, bias=False, compute_dtype=torch.float16,
                         quant_type="nf4").to(device)
with torch.inference_mode():
    assert torch.isfinite(layer(x)).all()
torch.cuda.synchronize()
report = {"python": platform.python_version(), "packages": versions,
          "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
          "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
          "cuda_and_bnb_nf4_check": "passed"}
Path(sys.argv[2]).write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print(json.dumps(report, ensure_ascii=False, indent=2))
""", PINS_PATH, DRIVE_RUN / "environment.json", log_name="environment.log")


## 5. 코드와 Train을 Colab 로컬 디스크로 복사
Drive의 많은 이미지 파일을 학습 중 반복해서 읽지 않도록 `/content`로 복사합니다.
큰 데이터셋은 폴더보다 ZIP으로 올리면 준비가 빠를 수 있습니다.
학습 코드 사본은 Drive의 실험 폴더에도 보관합니다.

In [ ]:
import stat
import zipfile

ignored = shutil.ignore_patterns(".venv", ".venv-*", "__pycache__", ".cache", ".pytest_cache",
                                ".git", "models", "outputs", "sample_submission", "colab_runs",
                                "data", "Train", "Test", "*.zip", "*.ipynb", ".ipynb_checkpoints")
shutil.copytree(DRIVE_STARTERKIT, LOCAL_KIT, dirs_exist_ok=True, ignore=ignored)
shutil.copytree(LOCAL_KIT, DRIVE_RUN / "source", dirs_exist_ok=True)

STAGED_DATA = LOCAL_ROOT / "data"
marker = LOCAL_ROOT / "data-ready.json"
source_stat = TRAIN_SOURCE.stat()
source_stamp = {"path": str(TRAIN_SOURCE), "size": source_stat.st_size,
                "mtime_ns": source_stat.st_mtime_ns}
if marker.is_file():
    saved = json.loads(marker.read_text())
    if saved["source"] != source_stamp:
        raise RuntimeError("입력 데이터가 바뀌었습니다. 2번 셀에서 새 실험을 만드세요.")
    DATA_DIR = Path(saved["data_dir"])
    print("복사 완료된 데이터 재사용:", DATA_DIR)
else:
    if STAGED_DATA.exists():
        raise RuntimeError(f"미완료 복사 폴더가 있습니다. 2번 셀에서 새 실험을 만드세요: {STAGED_DATA}")
    free_bytes = shutil.disk_usage(LOCAL_ROOT).free
    reserve = 4 * 1024**3  # 인코더, export, ZIP 생성 공간
    if TRAIN_SOURCE.is_dir():
        size = sum(p.stat().st_size for p in TRAIN_SOURCE.rglob("*") if p.is_file())
        if size + reserve > free_bytes:
            raise RuntimeError("Colab 로컬 디스크의 여유 공간이 부족합니다.")
        print(f"Train 폴더 복사 시작: {size / 2**30:.2f} GiB", flush=True)
        shutil.copytree(TRAIN_SOURCE, STAGED_DATA / "Train")
    elif zipfile.is_zipfile(TRAIN_SOURCE):
        # Drive의 압축 파일은 한 번만 읽고 로컬에서 압축을 해제합니다.
        with zipfile.ZipFile(TRAIN_SOURCE) as archive:
            expanded = sum(item.file_size for item in archive.infolist())
        if source_stat.st_size + expanded + reserve > free_bytes:
            raise RuntimeError("ZIP 복사·압축 해제에 필요한 로컬 디스크 공간이 부족합니다.")
        local_archive = LOCAL_ROOT / "train-input.zip"
        shutil.copyfile(TRAIN_SOURCE, local_archive)
        STAGED_DATA.mkdir(parents=True)
        root = STAGED_DATA.resolve()
        with zipfile.ZipFile(local_archive) as archive:
            for item in archive.infolist():
                destination = (root / item.filename).resolve()
                if not destination.is_relative_to(root) or stat.S_ISLNK(item.external_attr >> 16):
                    raise ValueError(f"지원하지 않는 ZIP 경로: {item.filename}")
            archive.extractall(root)
        local_archive.unlink()
    else:
        raise ValueError("TRAIN_SOURCE에는 Train 폴더 또는 ZIP 파일을 지정하세요.")
    candidates = [STAGED_DATA, STAGED_DATA / "Train", STAGED_DATA / "data/Train"]
    matches = [p for p in candidates if any(p.glob("episode_*/description.txt"))]
    if len(matches) != 1:
        raise ValueError("Train 폴더 구조를 확인하세요: episode_*/description.txt와 images/가 필요합니다.")
    DATA_DIR = matches[0]
    marker.write_text(json.dumps({"source": source_stamp, "data_dir": str(DATA_DIR)}, indent=2))
print("학습 데이터:", DATA_DIR)
print("로컬 코드:", LOCAL_KIT)


## 6. 학습 설정과 인코더 준비
`models/dinov2`, `models/mpnet`이 Drive에 있으면 복사하고, 없으면 공개된 두 인코더를 다운로드합니다.
다운로드는 준비 단계에서만 필요합니다. 이후 학습·검증은 로컬 모델을 사용합니다.
Train을 분할했을 때 각 split에 서로 다른 설명이 두 개 이상 있는지도 미리 검사합니다.

In [ ]:
CONFIG_PATH = LOCAL_KIT / "config-colab.json"
config = json.loads((LOCAL_KIT / "config.json").read_text(encoding="utf-8"))
config.update({"data_dir": str(DATA_DIR), "image_model_dir": str(LOCAL_KIT / "models/dinov2"),
               "text_model_dir": str(LOCAL_KIT / "models/mpnet"),
               "output_dir": str(CHECKPOINT_DIR), "epochs": 3 if SMOKE_TEST else EPOCHS,
               "image_batch_size": IMAGE_BATCH_SIZE, "text_batch_size": TEXT_BATCH_SIZE,
               "cpu_threads": 2})
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
shutil.copyfile(CONFIG_PATH, DRIVE_RUN / "config-colab.json")
run_python(r"""
import json, random, sys
from dataset import load_episodes, split_indices
cfg = json.load(open(sys.argv[1]))
episodes = load_episodes(cfg["data_dir"])
total = len(episodes)
if sys.argv[2] == "True":
    indices = random.Random(cfg["seed"]).sample(range(total), min(12, total))
    episodes = [episodes[i] for i in sorted(indices)]
train, valid = split_indices(episodes, cfg["validation_fraction"], cfg["seed"])
for name, indices in (("train", train), ("valid", valid)):
    assert len({episodes[i].description for i in indices}) >= 2, f"{name}: 서로 다른 설명이 두 개 이상 필요합니다."
print(f"데이터 확인 완료: 전체 {total} / 사용 {len(episodes)} / 학습 {len(train)} / 검증 {len(valid)}")
""", CONFIG_PATH, str(SMOKE_TEST), log_name="data-check.log")

model_files = {"dinov2": ["config.json", "model.safetensors", "preprocessor_config.json"],
               "mpnet": ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]}
source_models = DRIVE_STARTERKIT / "models"
models_complete = all((source_models / name / file).is_file()
                      for name, files in model_files.items() for file in files)
model_args = ["--config", CONFIG_PATH]
if models_complete:
    model_args += ["--source-root", source_models]
    print("Drive의 인코더를 복사합니다.")
else:
    print("허용된 DINOv2 / MPNet 인코더를 다운로드합니다.")
run_script("download_models.py", *model_args, log_name="models.log")
print(json.dumps(config, ensure_ascii=False, indent=2))


## 7. 학습
전체 프레임의 인코더 특징을 먼저 추출한 뒤 MLP를 학습합니다.
검증 BCE가 가장 낮은 가중치는 Drive의 `training/best.pt`에 저장됩니다.
`history.json`, `splits.json`, 특징 캐시와 로그도 실험 폴더에 보관됩니다.

**세션이 끊긴 경우:** Drive에 저장을 마친 파일은 남지만 `/content`는 사라질 수 있습니다.
현재 `train.py`는 optimizer 상태를 복구하는 이어 학습을 지원하지 않습니다.
새 학습은 2번 셀부터 새 실험으로 실행하세요. 학습 셀을 같은 실험에서 재실행하면
기존 체크포인트를 덮어쓰지 않고 오류로 중단합니다.

In [ ]:
args = ["--config", CONFIG_PATH, "--device", "cuda:0"]
if SMOKE_TEST:
    args += ["--max-episodes", "12"]
run_script("train.py", *args, log_name="train.log",
           extra_env={"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
CHECKPOINT = CHECKPOINT_DIR / "best.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError("best.pt가 생성되지 않았습니다. train.log를 확인하세요.")
print("선택된 체크포인트:", CHECKPOINT)


## 8. 로컬 검증
학습에서 저장한 검증 split과 특징 캐시로 최적 체크포인트를 평가합니다.
출력되는 `binary_accuracy`, `pair_accuracy`, BCE는 **random negative를 이용한 학습용 지표**입니다.
대회 리더보드의 5개 선지 순위 기반 score를 측정하는 단계가 아닙니다.

In [ ]:
run_script("evaluate.py", "--checkpoint", CHECKPOINT, "--device", "cuda:0",
           log_name="evaluate.log",
           extra_env={"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
history = json.loads((CHECKPOINT_DIR / "history.json").read_text())
print("epoch | train BCE | valid BCE | valid pair accuracy")
for row in history:
    print(f"{row['epoch']:5d} | {row['train']['bce']:.5f}   | {row['valid']['bce']:.5f}   | {row['valid']['pair_accuracy']:.4f}")


## 9. 제출 디렉토리 생성 및 오프라인 추론 확인
직접 학습한 MLP와 인코더·토크나이저를 함께 export합니다.
Train의 입력으로 모든 선지 ID가 한 번씩 반환되는지, 반복 추론이 일관적인지 검사합니다.
이 검사는 인터페이스 확인이며 전체 대회 데이터의 성능·시간 제한 통과를 보장하지 않습니다.

In [ ]:
run_script("export.py", "--checkpoint", CHECKPOINT, "--output-dir", EXPORT_DIR,
           log_name="export.log",
           extra_env={"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
# 별도 CWD에서 실행하여 export된 모듈을 실제로 사용합니다.
validator = LOCAL_ROOT / "validate-export.py"
validation_code = r"""
import sys
from pathlib import Path
artifact, kit, data = map(Path, sys.argv[1:4])
sys.path.insert(0, str(artifact))
sys.path.insert(1, str(kit))
import baseline_model
assert Path(baseline_model.__file__).parent == artifact
from validate import check_direct
check_direct(artifact, data, "cuda:0")
"""
validator.write_text(validation_code, encoding="utf-8")
run_command([PYTHON, "-I", validator, EXPORT_DIR, LOCAL_KIT, DATA_DIR],
            cwd=LOCAL_ROOT, log_name="validate.log",
            extra_env={"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
                       "HF_HOME": str(LOCAL_ROOT / "empty-validation-cache")})


## 10. ZIP 생성과 Drive 저장
ZIP 최상위에 `submission.py`가 오도록 패키징하고 크기·파일 구조·체크섬을 확인합니다.
생성한 ZIP만 대회 사이트에 제출하세요. `starterkit` 폴더 전체를 압축하지 않습니다.

ZIP과 압축 해제 후 용량은 각각 **1 GiB 이하**인지 확인합니다(2026-10-31 채점 환경 기준).
기존 `package.py`에 표시되는 256 MiB 관련 문구는 이전 서버 기본값 안내입니다.
최종 제출 제한과 참가 규정은 대회 사이트의 최신 안내를 따르세요.

In [ ]:
run_script("package.py", "--submission-dir", EXPORT_DIR, "--output", LOCAL_ZIP,
           "--force", log_name="package.log")
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    names = archive.namelist()
    required_in_zip = {"submission.py", "submission.json", "weights/mlp.pt",
                       "models/dinov2/model.safetensors", "models/mpnet/model.safetensors"}
    if not required_in_zip.issubset(names):
        raise ValueError("제출 ZIP 최상위 구조가 올바르지 않습니다.")
    if json.loads(archive.read("submission.json")) != {"api_version": 2}:
        raise ValueError("submission.json의 api_version을 확인하세요.")
    expanded_bytes = sum(info.file_size for info in archive.infolist())
    if expanded_bytes > 1024**3 or LOCAL_ZIP.stat().st_size > 1024**3 or len(names) > 10000:
        raise ValueError("제출 ZIP 용량 또는 파일 수 제한을 초과했습니다.")
    bad = archive.testzip()
    if bad:
        raise ValueError(f"ZIP 무결성 검사 실패: {bad}")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()

local_sha256 = sha256_file(LOCAL_ZIP)
DRIVE_ZIP = DRIVE_RUN / LOCAL_ZIP.name
partial = DRIVE_ZIP.with_suffix(".zip.partial")
shutil.copyfile(LOCAL_ZIP, partial)
if sha256_file(partial) != local_sha256:
    raise IOError("Drive 복사본의 체크섬이 다릅니다. 이 셀을 다시 실행하세요.")
partial.replace(DRIVE_ZIP)
summary = {"run": RUN_NAME, "smoke_test": SMOKE_TEST,
           "checkpoint": str(CHECKPOINT), "submission_zip": str(DRIVE_ZIP),
           "zip_bytes": DRIVE_ZIP.stat().st_size, "expanded_bytes": expanded_bytes,
           "sha256": local_sha256, "files": len(names)}
(DRIVE_RUN / "submission-summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
if SMOKE_TEST:
    print("이 ZIP은 소규모 기능 확인용 학습 결과입니다. 전체 학습은 SMOKE_TEST=False로 새 실험을 실행하세요.")
print("완료! Google Drive에서 위 ZIP을 내려받아 대회 사이트에 제출하세요.")
if DOWNLOAD_ZIP_TO_COMPUTER:
    from google.colab import files
    files.download(str(LOCAL_ZIP))


## 코드 수정과 재실행
- **모델·학습 코드 변경:** Drive의 `baseline_model.py`, `train.py`, `submission.py` 등을 수정한 뒤
  2번 셀부터 새 실험을 시작합니다. 실행에 사용한 코드 사본은 실험 폴더의 `source/`에 남습니다.
- **추가 실험 코드:** `run_python("import torch; print(torch.__version__)")` 또는
  `run_script("내스크립트.py", "--인자", "값")`로 대회 버전 환경에서 실행합니다.
- **CUDA out of memory:** 2번 셀에서 이미지·텍스트 배치 크기를 낮춰 새 실험을 시작합니다.
  이 baseline은 모든 프레임을 사용하므로 이미지 배치를 낮춰도 프레임을 버리지 않습니다.
- **Drive 경로 오류:** `MyDrive` 아래 실제 경로와 `episode_*/description.txt`, `images/` 구조를 확인합니다.
- **export 셀 재실행 오류:** 기존 제출 폴더를 자동으로 덮어쓰지 않습니다.
  `EXPORT_DIR`을 새로운 로컬 경로로 지정하고 9번 셀부터 실행할 수 있습니다.

환경 설정 참고: [Colab FAQ](https://research.google.com/colaboratory/faq.html),
[PyTorch 버전별 설치 안내](https://pytorch.org/get-started/previous-versions/),
[uv Python 설치 안내](https://docs.astral.sh/uv/guides/install-python/).